## Import Packages

In [1]:
import pandas as pd
import ast
import re
from datasets import load_dataset

## Extract arXiv IDs

In [5]:
# Load OpenAlex dataset
openalex_df = pd.read_csv("https://www.dropbox.com/scl/fi/53pvu6pxd3z1mbv2tj26f/openalex_works.csv?rlkey=tgrpo8ztw9wfdw2qrsnengya5&st=ly8iy6vt&dl=1")

def extract_arxiv_id(locations):
    if pd.isna(locations):
        return None

    # Convert CSV string back into a Python list
    if isinstance(locations, str):
        locations = ast.literal_eval(locations)

    for location in locations:
        url = location.get("landing_page_url") or location.get("pdf_url") or ""

        match = re.search(r"arxiv\.org/(?:abs|pdf)/([^?#]+)", url)
        if match:
            arxiv_id = match.group(1)
            return re.sub(r"v\d+$", "", arxiv_id)
    return None

openalex_df["arxiv_id"] = openalex_df["locations"].apply(extract_arxiv_id)

# Remove papers with missing and/or duplicated arXiv IDs
openalex_df.dropna(subset=["arxiv_id"], inplace=True)
openalex_df.drop_duplicates(subset=["arxiv_id"], inplace=True)

openalex_ids = openalex_df["arxiv_id"]
print(f"Extracted {len(openalex_ids):,} arXiv IDs")

Extracted 122,136 arXiv IDs


## Match Papers from Hugging Face

In [6]:
# Load the (entire) arXiv dataset from Hugging Face
dataset = load_dataset("open-index/open-arxiv", split="train", streaming=True)

arxiv_records = []

for paper in dataset:
    arxiv_id = re.sub(r"v\d+$", "", str(paper["id"]))

    if arxiv_id in openalex_ids:
        arxiv_records.append({"arxiv_id": arxiv_id,
                              "categories": paper["categories"],
                              "comments": paper["comments"]})

arxiv_df = pd.DataFrame(arxiv_records)

# Remove papers with duplicate arXiv IDs
arxiv_df.drop_duplicates(subset=["arxiv_id"], inplace=True)

print(f"Matched {len(arxiv_df):,} papers")

README.md:   0%|          | 0.00/19.9k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/417 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/417 [00:00<?, ?it/s]

KeyboardInterrupt: 

## Extract Papers with Code

In [7]:
# Load Papers with Code dataset
dataset_code = load_dataset("pwc-archive/links-between-paper-and-code", split="train")

pwc_df = dataset_code.to_pandas()

# Normalize arXiv IDs to be consistent
def normalize_arxiv_id(arxiv_id):
    if pd.isna(arxiv_id):
        return None

    arxiv_id = str(arxiv_id).strip()
    arxiv_id = re.sub(r"\.pdf$", "", arxiv_id)
    arxiv_id = re.sub(r"v\d+$", "", arxiv_id)

    return arxiv_id

pwc_df["arxiv_id"] = pwc_df["paper_arxiv_id"].apply(normalize_arxiv_id)

# Remove papers with no repo_url
pwc_df = pwc_df[pwc_df["repo_url"].str.strip() != ""]

# Remove nulls and duplicate using (arxiv_id, repo_url) pairs
pwc_df.dropna(subset=["arxiv_id", "repo_url"], inplace=True)
pwc_df.drop_duplicates(subset=["arxiv_id", "repo_url"], inplace=True)

# Group by each paper's unique arXiv id
pwc_features = pwc_df.groupby("arxiv_id").agg(num_repositories=("repo_url", "nunique")).reset_index()

# Create binary feature (Every paper appears here has code)
pwc_features["has_code"] = 1

print(f"Papers with code: {len(pwc_features):,}")

README.md:   0%|          | 0.00/826 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 41.4MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/300161 [00:00<?, ? examples/s]

Papers with code: 207,415


**Note**: A paper without a Papers with Code match does not necessarily mean it has no publicly available code. It only means the archive does not contain a matching repository record. Therefore, `has_code` indicates whether a repository is linked in the Papers with Code archive, rather than whether code actually exists.

Also, repository links may have been added after the paper's initial publication.

## Join Dataset

In [8]:
# Left join to preserve all OpenAlex papers
# final_df = openalex_df.merge(arxiv_df, on="arxiv_id",how="left")

# Join Papers with Code features
# final_df = final_df.merge(pwc_features, on="arxiv_id", how="left")
final_df = openalex_df.merge(pwc_features, on="arxiv_id", how="left")

# Papers without repository matches receive zero
final_df["has_code"] = final_df["has_code"].fillna(0).astype(int)
final_df["num_repositories"] = (final_df["num_repositories"].fillna(0).astype(int))

# Remove any nulls
final_df.dropna(inplace=True)

print(f"Final dataset: {len(final_df):,} papers")
print(f"Papers with code: {final_df['has_code'].sum():,}")

print(final_df.info())

Final dataset: 108,228 papers
Papers with code: 25,791
<class 'pandas.core.frame.DataFrame'>
Index: 108228 entries, 0 to 122134
Data columns (total 23 columns):
 #   Column                       Non-Null Count   Dtype 
---  ------                       --------------   ----- 
 0   id                           108228 non-null  object
 1   doi                          108228 non-null  object
 2   title                        108228 non-null  object
 3   abstract_inverted_index      108228 non-null  object
 4   publication_date             108228 non-null  object
 5   publication_year             108228 non-null  int64 
 6   counts_by_year               108228 non-null  object
 7   authorships                  108228 non-null  object
 8   institutions_distinct_count  108228 non-null  int64 
 9   countries_distinct_count     108228 non-null  int64 
 10  referenced_works_count       108228 non-null  int64 
 11  referenced_works             108228 non-null  object
 12  primary_topic         